# Driver Drowsiness Detection — Week 5
### Two-Model Training — Eye State + Yawn Detection
NTCC | Amity School of Engineering & Technology | June 2026

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

DRIVE  = '/content/drive/MyDrive/NTCC_Drowsiness_Project'
TRAIN  = f'{DRIVE}/data/train'
RES    = f'{DRIVE}/results'
MODELS = f'{DRIVE}/models'
os.makedirs(RES, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

BATCH = 32
SEED  = 42

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

Realised after Week 4's baseline attempt that the dataset isn't really one 4-class problem — Closed/Open are eye crops, yawn/no_yawn are full face photos. Training one model on both was why accuracy stayed around 33-45%. Splitting into two separate models below.

In [ ]:
def load_images(classes, label_map, size):
    X, y = [], []
    for cls in classes:
        path = os.path.join(TRAIN, cls)
        imgs = [f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for fname in imgs:
            img = cv2.imread(os.path.join(path, fname))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (size, size))
            X.append(img)
            y.append(label_map[cls])
        print(f'  {cls}: {len(imgs)} loaded')
    X = preprocess_input(np.array(X, dtype='float32'))
    y = np.array(y, dtype='int32')
    print(f'  shape: {X.shape}')
    return X, y

def make_split(X, y, n):
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
    Xv, Xte, yv, yte = train_test_split(Xtmp, ytmp, test_size=0.50, random_state=SEED, stratify=ytmp)
    return Xtr, to_categorical(ytr,n), Xv, to_categorical(yv,n), Xte, to_categorical(yte,n)

def build_mobilenet(size, n, name):
    base = MobileNetV2(input_shape=(size, size, 3), include_top=False, weights='imagenet')
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(n, activation='softmax')(x)
    return Model(inputs=base.input, outputs=out, name=name), base

## Model 1 — Eye State (Closed vs Open)

In [ ]:
print('Loading eye images...')
X_eye, y_eye = load_images(['Closed','Open'], {'Closed':0,'Open':1}, 96)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for row, label in enumerate([0, 1]):
    name = ['Closed','Open'][label]
    for col, idx in enumerate(np.where(y_eye == label)[0][:5]):
        img = X_eye[idx].copy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 2: axes[row][col].set_title(name, fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RES}/eye_samples.png', dpi=150)
plt.show()

In [ ]:
Xe_tr, ye_tr, Xe_val, ye_val, Xe_te, ye_te = make_split(X_eye, y_eye, 2)
print(f'Train: {Xe_tr.shape[0]}  Val: {Xe_val.shape[0]}  Test: {Xe_te.shape[0]}')

In [ ]:
eye_model, eye_base = build_mobilenet(96, 2, 'Eye_State_MobileNetV2')
print(f'Params: {eye_model.count_params():,}')

In [ ]:
aug = ImageDataGenerator(rotation_range=10, horizontal_flip=True, brightness_range=[0.8,1.2], zoom_range=0.1)
tr_gen  = aug.flow(Xe_tr, ye_tr, batch_size=BATCH, shuffle=True)
val_gen = ImageDataGenerator().flow(Xe_val, ye_val, batch_size=BATCH)

eye_model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
print('Phase 1 — training head, base frozen...')
h1 = eye_model.fit(
    tr_gen, steps_per_epoch=len(Xe_tr)//BATCH, epochs=20,
    validation_data=val_gen, validation_steps=len(Xe_val)//BATCH,
    callbacks=[
        ModelCheckpoint(f'{MODELS}/eye_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        CSVLogger(f'{RES}/eye_p1.csv')
    ], verbose=1
)
print(f'Phase 1 best val_acc: {max(h1.history["val_accuracy"]):.4f}')

In [ ]:
eye_base.trainable = True
for layer in eye_base.layers[:-30]:
    layer.trainable = False
eye_model.compile(Adam(1e-4), 'categorical_crossentropy', ['accuracy'])

print('Phase 2 — fine-tuning last 30 layers...')
h2 = eye_model.fit(
    tr_gen, steps_per_epoch=len(Xe_tr)//BATCH, epochs=20,
    validation_data=val_gen, validation_steps=len(Xe_val)//BATCH,
    callbacks=[
        ModelCheckpoint(f'{MODELS}/eye_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        CSVLogger(f'{RES}/eye_p2.csv')
    ], verbose=1
)
print(f'Phase 2 best val_acc: {max(h2.history["val_accuracy"]):.4f}')

In [ ]:
best_eye = load_model(f'{MODELS}/eye_model.h5')
eye_loss, eye_acc = best_eye.evaluate(Xe_te, ye_te, verbose=0)
print(f'Eye Model — Test Accuracy: {eye_acc*100:.2f}%')

ye_pred = np.argmax(best_eye.predict(Xe_te, verbose=0), axis=1)
ye_true = np.argmax(ye_te, axis=1)

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(ye_true, ye_pred), display_labels=['Closed','Open']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Eye State — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{RES}/Eye_State_confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(ye_true, ye_pred, target_names=['Closed','Open'], digits=4))

## Model 2 — Yawn Detection (yawn vs no_yawn)

In [ ]:
print('Loading face images...')
X_face, y_face = load_images(['yawn','no_yawn'], {'yawn':0,'no_yawn':1}, 96)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for row, label in enumerate([0, 1]):
    name = ['yawn','no_yawn'][label]
    for col, idx in enumerate(np.where(y_face == label)[0][:5]):
        img = X_face[idx].copy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 2: axes[row][col].set_title(name, fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RES}/face_samples.png', dpi=150)
plt.show()

In [ ]:
Xf_tr, yf_tr, Xf_val, yf_val, Xf_te, yf_te = make_split(X_face, y_face, 2)
print(f'Train: {Xf_tr.shape[0]}  Val: {Xf_val.shape[0]}  Test: {Xf_te.shape[0]}')

In [ ]:
face_model, face_base = build_mobilenet(96, 2, 'Yawn_Detection_MobileNetV2')

f_aug = ImageDataGenerator(rotation_range=8, horizontal_flip=True, brightness_range=[0.8,1.2], zoom_range=0.1)
ftr_gen  = f_aug.flow(Xf_tr, yf_tr, batch_size=BATCH, shuffle=True)
fval_gen = ImageDataGenerator().flow(Xf_val, yf_val, batch_size=BATCH)

face_model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
print('Phase 1 — training head, base frozen...')
hf1 = face_model.fit(
    ftr_gen, steps_per_epoch=len(Xf_tr)//BATCH, epochs=20,
    validation_data=fval_gen, validation_steps=len(Xf_val)//BATCH,
    callbacks=[
        ModelCheckpoint(f'{MODELS}/face_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        CSVLogger(f'{RES}/face_p1.csv')
    ], verbose=1
)
print(f'Phase 1 best val_acc: {max(hf1.history["val_accuracy"]):.4f}')

In [ ]:
face_base.trainable = True
for layer in face_base.layers[:-30]:
    layer.trainable = False
face_model.compile(Adam(1e-4), 'categorical_crossentropy', ['accuracy'])

print('Phase 2 — fine-tuning last 30 layers...')
hf2 = face_model.fit(
    ftr_gen, steps_per_epoch=len(Xf_tr)//BATCH, epochs=20,
    validation_data=fval_gen, validation_steps=len(Xf_val)//BATCH,
    callbacks=[
        ModelCheckpoint(f'{MODELS}/face_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        CSVLogger(f'{RES}/face_p2.csv')
    ], verbose=1
)
print(f'Phase 2 best val_acc: {max(hf2.history["val_accuracy"]):.4f}')

In [ ]:
best_face = load_model(f'{MODELS}/face_model.h5')
face_loss, face_acc = best_face.evaluate(Xf_te, yf_te, verbose=0)
print(f'Yawn Model — Test Accuracy: {face_acc*100:.2f}%')

yf_pred = np.argmax(best_face.predict(Xf_te, verbose=0), axis=1)
yf_true = np.argmax(yf_te, axis=1)

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(yf_true, yf_pred), display_labels=['yawn','no_yawn']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Yawn Detection — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{RES}/Yawn_Detection_confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(yf_true, yf_pred, target_names=['yawn','no_yawn'], digits=4))

In [ ]:
print('='*50)
print('WEEK 5 RESULTS')
print('='*50)
print(f'Eye model  test accuracy : {eye_acc*100:.2f}%')
print(f'Yawn model test accuracy : {face_acc*100:.2f}%')
print()
print('Yawn model is the harder task — subtle mouth differences in a full')
print('face scene vs an obvious eye crop difference. Plan for Week 6:')
print('extended fine-tuning on the yawn model specifically.')
print('='*50)